# 19c - Re-fly the landings whose Matchpoint burn failed (Colab version)

Seed 14 of the 125 m runs failed because the Matchpoint targeting solve did not converge (first time-of-flight guess had no real root from about 56 m up). The solver is fixed (`solve_ballistic_to_target`, commit ace1645). This notebook finds the failed landings in the Drive results of notebook 18c (`burn_converged` false, timeout, or delivery error above 1,000 m), moves those files to `superseded/` (kept, not deleted), and flies the same seeds again with the fixed solver. Every other landing is skipped because the run is resumable.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Approve the Drive popup when it appears. Expect about 10 to 30 minutes per re-flown landing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/touchdown/accel_sens'
print(os.listdir(OUT))

In [ ]:
!git clone -q https://github.com/aryanyaksh-art/touch_down.git /tmp/touch_down || (cd /tmp/touch_down && git pull -q)
!sh /tmp/touch_down/scripts/kaggle_setup.sh 2>&1 | tail -5

In [ ]:
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/fetch_data.py --global
!cd /tmp/touch_down && TOUCHDOWN_DATA=/tmp/data python scripts/build_wide_dtm.py --out-dir /tmp/data

In [ ]:
# list the failures first, then set them aside (moved to superseded/, not deleted)
!python /tmp/touch_down/scripts/find_failures.py {OUT}/accel0_burn1 {OUT}/accel0_burn0
!python /tmp/touch_down/scripts/find_failures.py {OUT}/accel0_burn1 {OUT}/accel0_burn0 --move

In [ ]:
# fly the missing seeds again (same commands as 18c; finished landings are skipped)
import subprocess
run = 'python /tmp/touch_down/scripts/kaggle_run.py --max-hours 6 --wide --seeds 0 50 --accel-scale 0.0'
cases = {'accel0_burn1': '', 'accel0_burn0': '--burn-error-scale 0.0'}
procs = [subprocess.Popen(f"{run} {extra} --out {OUT}/{name}", shell=True) for name, extra in cases.items()]
for p in procs:
    p.wait()
print('both flight processes finished', [p.returncode for p in procs])

In [ ]:
# check again: no failures should remain
!python /tmp/touch_down/scripts/find_failures.py {OUT}/accel0_burn1 {OUT}/accel0_burn0
!ls {OUT}/accel0_burn1/landing_*.json | wc -l; ls {OUT}/accel0_burn0/landing_*.json | wc -l